<a href="https://colab.research.google.com/github/saiKelkar/Property_Splats/blob/main/01_pipeline_test_mipnerf360_room.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc
!free -h

name, memory.total [MiB]
Tesla T4, 15360 MiB
8
               total        used        free      shared  buff/cache   available
Mem:            50Gi       1.5Gi        46Gi       2.4Mi       3.2Gi        49Gi
Swap:             0B          0B          0B


In [2]:
PROJECT = "mipnerf360_room"
ITERATIONS = 7000

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = f"/content/drive/MyDrive/splats/{PROJECT}"
WORK = f"/content/{PROJECT}"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
print("Drive folder:", DRIVE_DIR)
print("Work folder:", WORK)

Mounted at /content/drive
Drive folder: /content/drive/MyDrive/splats/mipnerf360_room
Work folder: /content/mipnerf360_room


In [3]:
!apt-get -qq update > /dev/null && apt-get -qq install -y colmap ffmpeg > /dev/null
!pip -q install nerfstudio
!ns-train --help > /dev/null && echo "nerfstudio OK"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is in

In [4]:
%cd /content
!wget -q --show-progress http://storage.googleapis.com/gresearch/refraw360/360_v2.zip
!unzip -q 360_v2.zip "room/*" -d {WORK}
!rm 360_v2.zip
!ls {WORK}/room

/content
360_v2.zip          100%[===================>]  11.67G   202MB/s    in 84s     
images	images_2  images_4  images_8  poses_bounds.npy	sparse


In [5]:
!ns-train splatfacto --output-dir {WORK}/outputs --vis tensorboard --max-num-iterations {ITERATIONS} colmap --data {WORK}/room --colmap-path sparse/0 --downscale-factor 4

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
I0000 00:00:1795154013.475040    5501 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
─────

In [6]:
!python -c "import torch, gsplat; print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| gsplat', gsplat.__version__)"
!ns-train splatfacto --output-dir {WORK}/outputs --vis tensorboard --max-num-iterations {ITERATIONS} colmap --data {WORK}/room --colmap-path sparse/0 --downscale-factor 4 > {WORK}/train_log.txt 2>&1; echo "EXIT CODE: $?"
!tail -n 40 {WORK}/train_log.txt

torch 2.14.1+cu130 | CUDA 13.0 | gsplat 1.4.0
EXIT CODE: 1
    ray_bundle, batch = self.datamanager.next_train(step)
                        ~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/nerfstudio/data/datamanagers/full_images_datamanager.py", line 352, in next_train
    data = self.cached_train[image_idx]
           ^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/functools.py", line 1025, in __get__
    val = self.func(instance)
  File "/usr/local/lib/python3.13/dist-packages/nerfstudio/data/datamanagers/full_images_datamanager.py", line 183, in cached_train
    return self._load_images("train", cache_images_device=self.config.cache_images)
           ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/nerfstudio/data/datamanagers/full_images_datamanager.py", line 232, in _load_images
    undistorted_images = list(
        track(
    ...<7 lines>...
        )
    )
  File "/usr/loca

In [ ]:
!MAX_JOBS=8 python -c "from gsplat.cuda._backend import _C; print('gsplat CUDA ready:', _C is not None)"; echo "EXIT CODE: $?"
!free -h

In [8]:
!MAX_JOBS=8 ns-train splatfacto --output-dir {WORK}/outputs --vis tensorboard --max-num-iterations {ITERATIONS} colmap --data {WORK}/room --colmap-path sparse/0 --downscale-factor 4 --downscale-rounding-mode ceil > {WORK}/train_log.txt 2>&1; echo "EXIT CODE: $?"
!tail -n 25 {WORK}/train_log.txt

EXIT CODE: 0
Step (% Done)                                                                       
--------------------                                                                
6910 (98.71%)       45.862 ms            4 s, 127.539 ms      10.74 M               
6920 (98.86%)       44.823 ms            3 s, 585.870 ms      10.90 M               
6930 (99.00%)       36.401 ms            2 s, 548.053 ms      11.12 M               
6940 (99.14%)       36.477 ms            2 s, 188.602 ms      11.10 M               
6950 (99.29%)       35.038 ms            1 s, 751.896 ms      11.58 M               
6960 (99.43%)       34.368 ms            1 s, 374.722 ms      11.80 M               
6970 (99.57%)       34.834 ms            1 s, 45.017 ms       11.64 M               
6980 (99.71%)       35.102 ms            702.043 ms           11.53 M               
6990 (99.86%)       35.518 ms            355.183 ms           11.40 M               
6999 (99.99%)                                       

In [10]:
import glob, shutil, os
configs = [c for c in glob.glob(f"{WORK}/outputs/**/config.yml", recursive=True)
           if glob.glob(os.path.join(os.path.dirname(c), "nerfstudio_models", "*.ckpt"))]
config = sorted(configs)[-1]
print("Using:", config)
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-export gaussian-splat --load-config {config} --output-dir "{DRIVE_DIR}/export"
shutil.copy(f"{WORK}/train_log.txt", DRIVE_DIR)
!ls -lh "{DRIVE_DIR}/export"

Using: /content/mipnerf360_room/outputs/unnamed/splatfacto/2026-11-20_060222/config.yml
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
I0000 00:00:1795155456.220017   12387 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Ple

In [11]:
import os, shutil
SPLATS = "/content/drive/MyDrive/splats"

# 1. The trained run (config + checkpoint)
run_dir = os.path.dirname(config)
shutil.copytree(run_dir, f"{DRIVE_DIR}/runs/{os.path.basename(run_dir)}", dirs_exist_ok=True)

# 2. The dataset, as one zip (skips the 12GB download next time)
shutil.make_archive(f"{WORK}/room", "zip", root_dir=WORK, base_dir="room")
shutil.copy(f"{WORK}/room.zip", DRIVE_DIR)

# 3. The compiled gsplat (skips the long compile next time)
shutil.make_archive("/content/torch_extensions_T4", "zip",
                    root_dir=os.path.expanduser("~/.cache"), base_dir="torch_extensions")
shutil.copy("/content/torch_extensions_T4.zip", SPLATS)

!ls -lh "{DRIVE_DIR}" "{SPLATS}"

/content/drive/MyDrive/splats:
total 8.2M
drwx------ 4 root root 4.0K Nov 20 06:20 mipnerf360_room
-rw------- 1 root root 8.2M Nov 20 06:20 torch_extensions_T4.zip

/content/drive/MyDrive/splats/mipnerf360_room:
total 1.3G
drwx------ 2 root root 4.0K Nov 20 06:18 export
-rw------- 1 root root 1.3G Nov 20 06:20 room.zip
drwx------ 3 root root 4.0K Nov 20 06:19 runs
-rw------- 1 root root 926K Nov 20 06:18 train_log.txt
